# 🏢 企业内部 API Agent 教程

本教程使用参照 **RAGFlow 代码组织方式** 的模块化项目，逐步实现一个调用企业内部 API 的 AI Agent。

## 📁 项目结构

```
company_agent/                    ← 项目根目录
├── __init__.py
├── tools/                        ← 工具层（对应 RAGFlow: agent/tools/）
│   ├── __init__.py
│   ├── base.py                   #   BaseTool 抽象基类 + ToolRegistry 注册表
│   ├── employees.py              #   GetEmployees 员工查询工具
│   ├── projects.py               #   GetProjects 项目查询工具
│   ├── metrics.py                #   GetMetrics 公司指标工具
│   └── search.py                 #   SearchCompany 内部搜索工具
├── llm/                          ← LLM 层（对应 RAGFlow: rag/llm/）
│   ├── __init__.py
│   └── client.py                 #   LLMClient 封装 DeepSeek-V4-Flash
├── prompts/                      ← 提示词层（对应 RAGFlow: rag/prompts/）
│   ├── __init__.py
│   ├── system.md                 #   系统提示词
│   └── next_step.md              #   下一步决策提示词
├── agent/                        ← Agent 层（对应 RAGFlow: agent/component/）
│   ├── __init__.py
│   └── agent.py                  #   AgentWithTools 推理循环
└── main.py                       ← 运行入口
```

## 🎯 学习目标
- 理解 RAGFlow 风格的分层架构设计
- 掌握如何通过 Function Calling 调用自定义 API
- 实现思维链（Chain of Thought）驱动的 Agent 推理
- 实现多 API 调用的并行执行
- 学会扩展新工具（新增 API）

## 🔑 使用的大模型
- **模型**: deepseek-v4-flash
- **API 端点**: `https://token-plan.cn-beijing.maas.aliyuncs.com/compatible-mode/v1`
- **兼容协议**: OpenAI Compatible

---
## 📦 Step 0: 环境准备

In [ ]:
# 安装依赖
!pip install openai httpx nest_asyncio 2>&1 | tail -3

In [ ]:
# 确保 company_agent 模块可导入
import sys, os
sys.path.insert(0, os.path.abspath("."))

from company_agent.tools import ToolRegistry
from company_agent.tools.employees import GetEmployees
from company_agent.tools.projects import GetProjects
from company_agent.tools.metrics import GetMetrics
from company_agent.tools.search import SearchCompany
from company_agent.llm import LLMClient
from company_agent.prompts import SYSTEM_PROMPT, NEXT_STEP_PROMPT, build_next_step_prompt
from company_agent.agent import AgentWithTools
from company_agent.main import create_registry

print("✅ 所有模块导入成功")

---
## 🏗️ Step 1: 工具层 — `company_agent/tools/`

> **对应 RAGFlow**: `agent/tools/`

每个工具是一个独立的 Python 文件，继承 `BaseTool`，实现 4 个属性/方法：

| 属性/方法 | 说明 | LLM 可见性 |
|-----------|------|------------|
| `name` | 工具的唯一标识 | ✅ 传给 LLM |
| `description` | 工具功能描述 | ✅ 传给 LLM |
| `parameters` | JSON Schema 参数定义 | ✅ 传给 LLM |
| `__call__` | 工具执行逻辑 | ❌ LLM 看不到 |

### 已实现的 4 个工具

- `tools/employees.py` — `GetEmployees` — 查询 `/api/employees`
- `tools/projects.py` — `GetProjects` — 查询 `/api/projects`
- `tools/metrics.py` — `GetMetrics` — 查询 `/api/metrics`
- `tools/search.py` — `SearchCompany` — 查询 `/api/search`

In [ ]:
# 查看工具基类源码（对应 RAGFlow: agent/tools/base.py）
from company_agent.tools.base import BaseTool, ToolRegistry
import inspect

print("=== BaseTool 抽象方法 ===")
for name, method in inspect.getmembers(BaseTool, predicate=inspect.isfunction):
    if not name.startswith("_"):
        print(f"  {name}")

print("\n=== ToolRegistry 方法 ===")
for name in dir(ToolRegistry):
    if not name.startswith("_"):
        print(f"  {name}")

In [ ]:
# 创建工具注册表并注册所有工具
API_BASE_URL = "http://localhost:8080"

registry = create_registry(api_base_url=API_BASE_URL)
print(f"✅ 已注册 {len(registry.get_all())} 个工具")
for tool in registry.get_all():
    print(f"   📝 {tool.name}: {tool.description[:40]}...")

In [ ]:
# 测试：直接调用工具验证 API 连通性
import asyncio

async def test_tools():
    print("🧪 测试 API 工具调用...\n")
    for name in ["get_employees", "get_projects", "get_metrics"]:
        tool = registry.get(name)
        result = await tool()
        print(f"--- {name} ---")
        print(result[:200])
        print()
    
    search = registry.get("search_company")
    result = await search(query="ai")
    print(f"--- search_company ---")
    print(result)

await test_tools()

---
## 🧠 Step 2: 提示词层 — `company_agent/prompts/`

> **对应 RAGFlow**: `rag/prompts/`

提示词使用独立的 `.md` 文件管理，而非硬编码在 Python 代码中。这样的好处是：
- 方便非技术人员修改提示词
- 版本控制清晰
- 支持多语言提示词

In [ ]:
# 查看提示词文件
from company_agent.prompts import SYSTEM_PROMPT, NEXT_STEP_PROMPT

print("=== 系统提示词 (system.md) ===")
print(SYSTEM_PROMPT[:300])
print(f"\n...总长度: {len(SYSTEM_PROMPT)} 字符")

print("\n=== 下一步提示词模板 (next_step.md) ===")
print(NEXT_STEP_PROMPT[:200])

In [ ]:
# 测试提示词构建
prompt = build_next_step_prompt("工程部有多少人？")
print("生成的提示词:")
print(prompt)

print("\n--- 带搜索历史的提示词 ---")
prompt_with_history = build_next_step_prompt(
    "工程部有多少人？",
    search_history="已查询：get_employees 返回 2 名员工"
)
print(prompt_with_history)

---
## ⚡ Step 3: LLM 层 — `company_agent/llm/`

> **对应 RAGFlow**: `rag/llm/chat_model.py`

封装 LLM 调用逻辑，支持 OpenAI 兼容协议的模型。

In [ ]:
# 初始化 LLM 客户端
llm = LLMClient()
print(f"模型: {llm.model}")
print(f"端点: {llm.base_url}")
print(f"模式: {llm.mode}")

In [ ]:
# 测试：直接调用 LLM（带工具定义）
async def test_llm():
    messages = [
        {"role": "system", "content": "你是公司助手。"},
        {"role": "user", "content": "工程部有哪些员工？"}
    ]
    tools = registry.to_openai_tools()
    print(f"工具数量: {len(tools)}")
    print(f"工具定义: {[t['function']['name'] for t in tools]}")
    
    text, tool_calls = await llm.chat(messages, tools=tools)
    
    if tool_calls:
        print(f"\nLLM 决定调用 {len(tool_calls)} 个工具:")
        for tc in tool_calls:
            print(f"  - {tc['name']}({tc['arguments']})")
    else:
        print(f"\nLLM 直接回答: {text[:100]}...")

await test_llm()

---
## 🔄 Step 4: Agent 层 — `company_agent/agent/`

> **对应 RAGFlow**: `agent/component/agent_with_tools.py:194-212`

Agent 推理循环实现 ReAct 模式：
```
思考 → 行动 → 观察 → 思考 → 行动 → 观察 → ... → 最终答案
```

关键流程：
1. 构建思维链提示词
2. 调用 LLM 获取决策
3. 如果返回 tool_calls → **并行执行**所有工具
4. 将结果加入对话历史，继续下一轮推理

In [ ]:
# 查看 Agent 类结构
import inspect

print("=== AgentWithTools 方法 ===")
for name, method in inspect.getmembers(AgentWithTools, predicate=inspect.isfunction):
    if not name.startswith("_"):
        sig = inspect.signature(method)
        print(f"  {name}{sig}")

---
## 🎬 Step 5: 完整运行示例

使用 `company_agent.main.run()` 便捷函数，一行代码运行 Agent。

### 示例 1: 项目 + 员工查询

In [ ]:
from company_agent.main import run
import nest_asyncio
try:
    nest_asyncio.apply()
except:
    pass

query1 = "我们公司有哪些进行中的项目？工程部有哪些员工？"
result1 = await run(query1, api_base_url=API_BASE_URL, max_steps=3)
print(f"\n✅ 示例 1 完成！")

### 示例 2: 指标 + 项目状态查询

In [ ]:
query2 = "公司 Q3 的营收情况如何？现在有哪些规划中的项目？"
result2 = await run(query2, api_base_url=API_BASE_URL, max_steps=3)
print(f"\n✅ 示例 2 完成！")

### 示例 3: 使用搜索工具

In [ ]:
query3 = "公司关于 AI 的项目有哪些？"
result3 = await run(query3, api_base_url=API_BASE_URL, max_steps=3)
print(f"\n✅ 示例 3 完成！")

---
## 📊 总结：如何扩展

### 添加新 API 工具（3 步）

**1. 在 `tools/` 目录下创建新文件**
```python
# tools/departments.py
from .base import BaseTool

class GetDepartments(BaseTool):
    def __init__(self, api_base_url="http://localhost:8080"):
        self.api_base_url = api_base_url
    
    @property
    def name(self): return "get_departments"
    
    @property
    def description(self): return "查询公司所有部门列表"
    
    @property
    def parameters(self):
        return {"type": "object", "properties": {}, "required": []}
    
    async def __call__(self, **kwargs) -> str:
        import httpx
        async with httpx.AsyncClient() as c:
            r = await c.get(f"{self.api_base_url}/api/departments")
            return str(r.json())
```

**2. 在 `main.py` 中注册**
```python
from .tools.departments import GetDepartments

def create_registry(...):
    registry = ToolRegistry()
    registry.register(GetDepartments(api_base_url=api_base_url))  # ← 新增
    # ... 其他工具
    return registry
```

**3. 完成！** LLM 会自动发现新工具并决定是否需要调用它。

### 切换 LLM 模型

```python
# 修改 llm/client.py 中的常量
BASE_URL = "https://your-api-endpoint/v1"
MODEL = "your-model-name"
```

或者运行时指定：
```python
llm = LLMClient(
    api_key="your-key",
    model="gpt-4o",
    base_url="https://api.openai.com/v1"
)
```

### 修改提示词

直接编辑 `prompts/system.md` 或 `prompts/next_step.md`，无需修改 Python 代码。